# 固定一次OOF动作标签路由重训
沿用原特征、C=.1、balanced逻辑回归、阈值.5，260条成熟标签一次拟合。独立成本账户保留原恢复和风险规则，比较原路由与每天执行。开发样本门不足仍为false，不晋升。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec,ACTION_FEATURES
from src.workflow.minute_pit_allocator import _dataset
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
import warnings
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,annual_paired_metrics,stationary_bootstrap_excess
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/tangent_router_refit_20261015';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256']
names=['inputs/plan','analysis/targets',*[f'analysis/native/{family}_{cost}' for family in ['candidate','soxx'] for cost in plan['slippage_bps']]]
strong,strong_audit=_source(root,plan['frozen_source'],names)
source,source_audit=_source(root,plan['source'],['inputs/frame']);controller,controller_audit=_source(root,plan['price_source'],['inputs/controller','inputs/plan']);action,action_audit=_source(root,plan['action_source'],['inputs/plan','controller/source_recovery_selection'])
label_objects,label_audit=_source(root,plan['label_source'],['analysis/labels'])
always,always_audit=_source(root,plan['always_source'],[f'analysis/native/strong_always_{v}' for v in plan['slippage_bps']])
for existing in [strong]:
 for key in ['tariff','initial_cash','start','end','as_of','slippage_bps','maximum_soxl_weight','minimum_cash_weight','maximum_drawdown']:assert existing['inputs/plan'][key]==plan[key]
assert controller['inputs/plan']['tariff']==plan['tariff']
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec'])
labels=label_objects['analysis/labels'].set_index('trade_date')
assert spec.logistic_c==.1 and spec.execute_probability_threshold==.5 and spec.minimum_training_rows==200
dataset=_dataset(labels[list(ACTION_FEATURES)],labels[['tiered_advantage']],pd.DataFrame(index=labels.index),labels[['label_available_at','training_qualified']],{'train':('2020-01-01','2022-12-31')})
prepared=dataset.prepare('train',col_set=['feature','label','meta'])
qualified=prepared[('meta','training_qualified')].astype(bool)&pd.to_datetime(prepared[('meta','label_available_at')]).lt(pd.Timestamp(plan['label_cutoff']))
train=prepared.loc[qualified];assert len(train)==260 and len(train)>=spec.minimum_training_rows
assert np.isfinite(train['feature'].to_numpy(float)).all()
y=train[('label','tiered_advantage')].gt(0);assert y.nunique()==2
model=make_pipeline(SimpleImputer(),StandardScaler(),LogisticRegression(C=spec.logistic_c,class_weight='balanced',max_iter=2000,random_state=42))
modified=dict(controller['inputs/controller'])
frame=source['inputs/frame'];frame=frame.loc[frame.signal_date.between(plan['start'],plan['end'])].copy()
books={};prefix=[]
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_audits={'strong':strong_audit,'labels':label_audit,'always':always_audit,'market':source_audit,'controller':controller_audit,'action':action_audit});rec.save_objects(artifact_path='sources',routing=(root/'src/workflow/stage1_open_action_value.py').read_text(),replay=(root/'src/backtest/full_path_tiered.py').read_text());rec.log_params(maximum_new_native_cases=3,new_fits=1)
 with warnings.catch_warnings(record=True) as caught:
  warnings.simplefilter('always',ConvergenceWarning);model.fit(train['feature'],y)
  if any(issubclass(w.category,ConvergenceWarning) for w in caught):raise ValueError('fixed logistic failed convergence')
 modified['frozen_open_action_value_model']=model
 training_audit={'training_rows':len(train),'positive_rows':int(y.sum()),'last_label_available_at':str(train[('meta','label_available_at')].max()),'cutoff':plan['label_cutoff'],'features':list(ACTION_FEATURES),'C':spec.logistic_c,'threshold':spec.execute_probability_threshold,'iterations':model[-1].n_iter_.tolist(),'development_rows_before_2022':154,'minimum_rows':200}
 rec.save_objects(artifact_path='models',router=model);rec.save_objects(artifact_path='inputs',training_audit=training_audit);rec.log_metrics(completed_model_fits=1)
for name,original in [('strong',strong)]:
 data=frame.copy();targets=original['analysis/targets']['candidate'].reindex(pd.DatetimeIndex(frame.signal_date));assert targets.notna().all().all()
 for col in targets:data[col]=targets[col].to_numpy()
 for cost in plan['slippage_bps']:
  replay=replay_full_path(data,models=None,plan=plan,slippage_bps=cost,controller=modified,recovery_spec=recovery,action_spec=spec,selection=action['controller/source_recovery_selection'])
  book=native_tiered_stage1(data,replay,provider_uri=provider,tariff=plan['tariff'],slippage_bps=cost,initial_cash=plan['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition)
  control=original[f'analysis/native/candidate_{cost}'];books[(name+'_refit',cost)]=book;books[(name+'_router',cost)]=control;books[('soxx',cost)]=strong[f'analysis/native/soxx_{cost}'];books[('strong_always',cost)]=always[f'analysis/native/strong_always_{cost}']
  assert book['authority'].trade_date.equals(control['authority'].trade_date)
  first=np.flatnonzero(book['authority'].execute_open_target.to_numpy()!=control['authority'].execute_open_target.to_numpy());stop=int(first[0]) if len(first) else len(frame)
  cols=['equity','ending_soxl_quantity','ending_soxx_quantity','ending_cash'];np.testing.assert_allclose(book['authority'][cols].iloc[:stop],control['authority'][cols].iloc[:stop],atol=1e-8,rtol=0)
  prefix.append({'family':name,'slippage_bps':cost,'unchanged_prefix_sessions':stop,'first_action_difference':str(frame.trade_date.iloc[stop].date()) if stop<len(frame) else None})
  rec.save_objects(artifact_path='analysis/native',**{f'{name}_refit_{cost}':book,f'{name}_router_{cost}':control});rec.log_metrics(completed_new_native_cases=len(prefix))
rows=[];annual=[];uncertainty=[]
for (family,cost),book in books.items():
 path=book['authority'];bench=books[('soxx',cost)]['authority'];relative=np.log1p(path.strategy_return)-np.log1p(bench.strategy_return)
 rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,bench),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'orders':len(book['orders']),'broker_fees_usd':path.broker_and_external_fees_usd.sum(),'slippage_usd':path.slippage_reserve_usd.sum(),'positive_126d_fraction':relative.rolling(126).sum().dropna().gt(0).mean()})
 annual.append(annual_paired_metrics(path,bench).assign(family=family,slippage_bps=cost))
 if family.endswith('_refit'):
  for comparator in ['soxx','strong_always',family.replace('_refit','_router')]:uncertainty.append({'family':family,'slippage_bps':cost,'comparator':comparator,**stationary_bootstrap_excess(path,books[(comparator,cost)]['authority'],mean_block=63,resamples=2000,alpha=.025,seed=20261015)})
table=pd.DataFrame(rows);annual=pd.concat(annual,ignore_index=True);uncertainty=pd.DataFrame(uncertainty);gates={}
for family in ['strong']:
 cand=table[table.family.eq(family+'_refit')].set_index('slippage_bps');control=table[table.family.eq(family+'_router')].set_index('slippage_bps')
 gates[family]={'beat_always_all_costs':bool(cand.terminal_wealth.gt(table[table.family.eq('strong_always')].set_index('slippage_bps').terminal_wealth).all()),'beat_paired_all_costs':bool(cand.terminal_wealth.gt(control.terminal_wealth).all()),'net_soxx_excess_all_costs':bool(cand.relative_wealth.gt(1).all()),'drawdown_30pct':bool(cand.max_drawdown_pct.ge(-30).all() and cand.monitored_drawdown_pct.ge(-30).all()),'cvar_no_worse':bool(cand.cvar5_pct.ge(cand.benchmark_cvar5_pct).all()),'rolling_70pct':bool(cand.positive_126d_fraction.ge(.7).all()),'annual_70pct':bool(annual[annual.family.eq(family+'_refit')].groupby('slippage_bps').relative_wealth.apply(lambda v:v.gt(1).mean()).ge(.7).all()),'bootstrap_positive':bool(uncertainty[uncertainty.family.eq(family+'_refit')].lower_bound_positive.all()),'independent_development':False,'qualified_forward':False}
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['analysis/native/*','inputs/*','sources/*']}
summary={'experiment':card.name,'decision':'not_promoted_fixed_tangent_router_refit','gates':gates,'new_fits':1,'new_native_cases':3,'objective_achieved':False,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':plan['scope']}
rec.save_objects(artifact_path='analysis',comparison=table,annual=annual,uncertainty=uncertainty,prefix=pd.DataFrame(prefix),summary=summary);rec.log_metrics(registered_artifact_count=17)
(card/'training_audit.json').write_text(json.dumps(training_audit,indent=2))
for name,value in [('backtest_benchmark_comparison',table),('annual_metrics',annual),('uncertainty',uncertainty),('prefix_verification',pd.DataFrame(prefix))]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/'research/opening_state_training_20261015/nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(a.rglob('*')) if p.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(table[['family','slippage_bps','return_pct','monitored_drawdown_pct']].to_string(index=False))


# OOF动作标签路由重训：优于旧路由，但不及简单执行对照

完成1次固定逻辑回归拟合、3套新增完整Qlib原生账户。260条成熟动作标签经DatasetH/DataHandlerLP读取并按严格2023-01-01 cutoff过滤；正类88条，最晚标签成熟2022-12-28 09:49。原22个路由特征、SimpleImputer/StandardScaler、C=0.10、balanced、random_state=42、max_iter=2000及执行概率阈值0.50不变；14次迭代收敛，没有调参或择优重训。

训练标签来自年度OOF配置和无学习型控制器的连续参考账户；外层仅替换长期小模型的开盘执行分类器，恢复模型、风险规则、EMA、仓位边界、费用合同全部保留。各成本独立结算自己的持仓、净值与风险状态。第一次开盘动作分歧均为2023-01-05，此前1个交易日的现金、股数、净值与原路由一致，原生Qlib费用及账簿对账通过。

本金10万美元，IBKR Pro Tiered冻结费用投影；2023-01-03信号至2026-10-09退出，共944区间。

| 执行规则 | 0bp净收益 | 2bp净收益 | 5bp净收益 | 2bp监控最大回撤 |
|---|---:|---:|---:|---:|
| 原路由 | +342.52% | +335.36% | +348.06% | 27.64% |
| OOF标签重训路由 | +357.52% | +352.01% | +350.64% | 27.71% |
| 每天执行风险投影目标 | +358.11% | +359.33% | +351.07% | 27.71% |
| SOXX | +387.75% | +387.73% | +387.70% | 43.39% |

重训路由在全部成本优于旧路由，但全部低于每天执行的简单对照和SOXX；监控回撤27.64%/27.71%/27.80%通过30%，超额、年度/滚动稳定及bootstrap门失败。不能只比较旧路由宣布成功，局部交易分类更换未显示超越简单执行的价值。

独立开发门仍为false：2020–2021仅154条动作标签，低于原200条训练门；未降低门槛、随机混洗或用2022标签训练其验证模型。外层944日已经反复用于研究，不是新独立OOS。训练参考不含学习型恢复而外层保留恢复，状态分布迁移风险仍存在；本次完整账户结果没有证实它可被这个分类器解决。价格、派息、费用与回撤仍为既定研究口径，非真实券商结算及全分钟最大回撤认证。

决策not_promoted_fixed_tangent_router_refit。固定路由重训路线关闭，不搜索C、概率阈值、特征子集、标签周期或重新采样。每天执行的长期模型仍仅是较好历史研究参考，不修改生产。当前证据把未解决问题收窄为：即便简化开盘执行，配置与风险组合仍缺少稳定净SOXX超额；后续需要先在时间隔离的训练/验证上证明配置层的信息增量，不能继续依靠优化小额交易跳过规则宣称达标。

训练模型、原生账户、年度/滚动/置信分析与来源身份保存于Qlib Recorder，NAS冷恢复逐对象哈希验证通过。1次训练和本轮回测完成不等于目标达成，稳定超额目标继续进行。
